## 1. Preprocessing Pipeline.
Define a unified preprocessing pipeline for all experiments at 256x256 RGB, compatible with ImageNet-pretrained models.
#### Rationale:
- Standarise inputs across CheXpert (grayscale, variable size) and TBX11K (RGB, 512x512).
- Enable fair comparison among custom CNN and transfer-learning baselines.
- Use ImageNet normalisation to align with pretrained weights.
- Allow an easy switch to 320x320 for the pre-registered sensitivity analysis.

In [3]:
# Define a reproducible preprocessing pipeline for training and validation/testing.
# Baseline: 256x256 RGB for all datasets; optional 320x320 sensitivity via IMG_SIZE variable.

from pathlib import Path
from torchvision import transforms as T

# Project root inferred from this notebook living in notebooks/
PROJECT_ROOT = Path("..").resolve()

# Dataset root paths.
CHEXPERT_PATH = PROJECT_ROOT / "data" / "chexpert"
TBX11K_PATH   = PROJECT_ROOT / "data" / "tbx11k"

# Toggle here for running the sensitivity analysis.
IMG_SIZE = 256 # Set to 320 for the sensitivity analysis.

# ImageNet normalisation (required for the MobileNetV2, EfficientNet-B0, ResNet-50 pretrained weights).
# Rescale pixel values between CheXpert and TBX11K datasets, so they are centred and comparable.
# During that training, inputs were normalised with specific mean and standard deviation values.
# Apply ImageNet normalisation to meet what pretrained filters expect.
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # MobileNetV2, EfficientNet-B0, and ResNet-50 were pretrained on
IMAGENET_STD  = [0.229, 0.224, 0.225] # ImageNet with these exact statistics.

# Training-time augmentations (mild, medically, sensible: avoid aggressive geometric distortions).
train_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),     # Standarise spatial size (IMG_SIZE X IMG_SIZE).
    T.Grayscale(num_output_channels=3), # CheXpert is L (1ch): replicate to RGB (3ch).
    T.RandomHorizontalFlip(p=0.5),      # Lungs are roughly symmetric; safe augmentation.
    T.RandomRotation(degrees=5),        # Small rotations to improve rebustness.
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),   # TBX11K is 512x512 RGB.
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation/test transforms (no stochastic augmentation). Only standarise size, convert to grey scale, to tenson, and normalise to match 
# ImageNet stats.
eval_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f'Preprocessing ready at {IMG_SIZE}x{IMG_SIZE} RGB.')
print(f'- CheXpert: grayscale -> RGB + normalisation.')
print(f'- TBX11K  : RGB + normalisation.')

# Clarify the modelling classes and their data sources (for logging/record-keeping).
print("\nClasses will be used for training: \n- 'Pneumonia' (CheXpert frontal-only)\n- 'Normal' (TBX11K) \n- 'TB' (TBX11K)")

Preprocessing ready at 256x256 RGB.
- CheXpert: grayscale -> RGB + normalisation.
- TBX11K  : RGB + normalisation.

Classes will be used for training: 
- 'Pneumonia' (CheXpert frontal-only)
- 'Normal' (TBX11K) 
- 'TB' (TBX11K)


# 2. Dataset scaffolding.
Build a unified three-class Dataset using RELATIVE paths, mirroring 01_data_eda.ipynb behavior.
Store relative paths and a 'Source' tag in a DataFrame; compose absolute paths lazily in `__getitem__`.
#### Target class order (fixed across the whole project):
- 0 : 'Pneumonia' (CheXpert frontal-only)
- 1 : 'TB'        (TBX11K)
- 2 : 'Normal'    (TBX11K)
#### Rationale:
- Keep paths relative to dataset roots for portability and reproducibility.
- Enforce RGB at load time to align with ImageNet-pretrained models.
- Avoid RAM preloading; open images on demand for efficiency.

In [4]:
# Build a unified three-class Dataset using RELATIVE path (mirror 01_data_eda.ipynb behavior).
# Store relative paths and a source tag in the DataFrame; compose absolute paths in __getitem__.
# Target classes: Pneumonia (CheXpert frontal-only), TB (TBX11K), Normal (TBX11K).

from pathlib import Path
from typing import List, Dict
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

# Stable label mapping used everywhere in training and evaluation.
# Fix the label order explicitely for all downstram components.
CLASS_TO_INDEX: Dict[str, int] = {"Pneumonia": 0, "TB": 1, "Normal": 2}
# Create the inverse mapping (index -> class) for readability, debugging, and plotting.
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}

def make_chexpert_pneumonia_frontal_df_rel(chexpert_root: Path) -> pd.DataFrame:
    """
    Build a DataFrame for CheXpert Pneumonia (frontal-only) with RELATIVE paths.
    Returns columns: ['rel_path', 'Class", 'Source'] with Class == 'Pneumonia' and Source == 'chexpert'.
    """
    # Load the CheXpert metadata (train.csv) which includes labels for 14 pathologies.
    meta = pd.read_csv(chexpert_root / "train.csv")

    # Keep only the file path and the "Pneumonia" column, discarding all other labels.
    df = meta[["Path", "Pneumonia"]].copy()

    # Remove the original dataset prefix ("CheXpert-v1.0-small/") to align with the extracted folder structure
    # in the repository. This makes paths portable.
    df["Path"] = df["Path"].str.replace("CheXpert-v1.0-small/", "", regex=False)

    # Keep only "frontal" views (exclude lateral X-rays) to ensure consistency with TBX11K, which only
    # provides frontal images.
    df = df[df["Path"].str.contains("frontal")]

    # Filter out only Pneumonia-positive cases (Pneumonia == 1).
    # Pneumonia == 1 : CheXpert’s binary label saying 'this image shows pneumonia.'
    # All other CheXpert cases (normal or other conditions) are discarded here.
    df = df[df["Pneumonia"] == 1]

    # Add a "Class" column explicitely set to "Pneumonia" for all rows.
    df["Class"] = "Pneumonia" 

    # Add a "Source" column so it is possible to distinguish between CheXpert and TBX11K.
    df["Source"] = "chexpert"

    # Rename "Path" to "rel_path" to make clear these are relative paths.
    df.rename(columns={"Path": "rel_path"}, inplace=True)

    # Return only the standarised columns: relative path, class name, and source dataset.
    return df[["rel_path", "Class", "Source"]]

def make_tbx11k_normal_tb_df_rel(tbx_root: Path, split_lists: List[str]) -> pd.DataFrame:
    """
    Build a DataFrame for TBX11K Normal/TB with RELATIVE paths.
    Returns columns: ['rel_path', 'Class', 'Source'] with Class in {'Normal', 'TB'} and Source == 'tbx11k'.
    """
    records = []

    # TBX11K does not come with a CSV. Instead, it provides split files (all_trainval.txt, all_test.txt, etc.),
    # where each line is a relative path to an image.
    for list_name in split_lists:
        with open(tbx_root / "lists" / list_name, "r") as f:
            for line in f:
                rel = line.strip()   # Read and clean the relative path.
                if not rel:
                    continue

                # Apply dataset-specific rules to assing class:
                # - If path starts with "sick" or "tb", OR ends with "_1.png", it is labelled TB.
                # - Otherwise, the image is considered Normal.
                cls = "TB" if rel.startswith(("sick", "tb")) or rel.endswith("_1.png") else "Normal"

                # Append a record to the list with relative path, assigned class, and source.
                records.append({"rel_path": rel, "Class": cls, "Source": "tbx11k"})

    # Convert the list of records into a clean DataFrame.
    return pd.DataFrame.from_records(records)

class CXRThreeClassDataset(Dataset):
    """
    Provide a unified three-class CXR dataset with the fixed label order:
      0 : 'Pneumonia' (CheXpert frontal-only)
      1 : 'TB'        (TBX11K)
      2 : 'Normal'    (TBX11K)
      
    Absolute paths are built at access time depending on the "Source" column.
    This class implements the standard PyTorch Dataset API (len, getitem).
    """

    def __init__(self, items: pd.DataFrame, transforms, chexpert_root: Path, tbx_root: Path):
        # Store the DataFrame of items (relative paths + labels + sources).
        # Resetting the index ensures sequencial indices from 0...N-1 for safe access.
        self.items = items.reset_index(drop=True)

        # Save preprocessing transforms (data augmetation / normalisation).
        self.transforms = transforms

        # Save root paths to both datasets: need to build absolute paths later.
        self.chexpert_root = chexpert_root
        self.tbx_root = tbx_root

    def __len__(self) -> int:
        # Required by PyTorch: return the total number of samples in the dataset.
        return len(self.items)

    def __getitem__(self, idx: int):
        # Fetch the metadata for a single sample (row from the DataFrame).
        row = self.items.iloc[idx]

        # Build the absolute file path:
        # - If from CheXpert, join with CheXpert root.
        # - If from TBX11K, join with tbx_root + "imgs" folder.
        abs_path = (self.chexpert_root / row["rel_path"]) if row["Source"] == "chexpert" else (self.tbx_root / "imgs" / row["rel_path"])

        # Load the image and convert to RGB explicitely.
        # This ensures grayscale CheXpert images become 3-channel, consistent with TBX11K and ImageNet.
        img = Image.open(abs_path).convert("RGB")

        # Apply preprocessing transforms if provided (resize, normalise, augment).
        if self.transforms is not None:
            img = self.transforms(img)

        # Convert the class string (e.g., "TB") into its numeric index (0/1/2).
        label = CLASS_TO_INDEX[row["Class"]]

        # Return a tuple (image tensor, label tensor) ready for PyTorch DataLoader.
        return img, torch.tensor(label, dtype=torch.long)

# --- Dataset Construction Pipeline ---

# Build the CheXpert Pneumonia-only DataFrame (frontal images, positive cases).
chexpert_pneu_rel_df = make_chexpert_pneumonia_frontal_df_rel(CHEXPERT_PATH)

# Build the TBX11K DataFrame for Normal and TB classes, using the all_trainval list.
tbx_pool_rel_df = make_tbx11k_normal_tb_df_rel(TBX11K_PATH, split_lists=["all_trainval.txt"])

# Combine CheXpert (Pneumonia) and TBX11K (TB + Normal) into one unified pool.
source_pool_rel = pd.concat([chexpert_pneu_rel_df, tbx_pool_rel_df], ignore_index=True)

# Count how many samples exist per class, then reorder to match the fixed mapping.
counts = source_pool_rel["Class"].value_counts()
ordered = pd.Series({k: counts.get(k, 0) for k in ["Pneumonia", "TB", "Normal"]})
print("Source pool counts (before split) [Pneumonia, TB, Normal]:")
print(ordered.to_string())

# Instantiate the unified PyTorch Dataset over the full pool.
# Note: transforms will be replace later with class-specific ones (CheXpert vs TBX11K),
# and add stratified cross-validation splits for fair training/evaluation.
train_dataset = CXRThreeClassDataset(
    items=source_pool_rel,
    transforms=train_transforms_tbx11k, # Placeholder: currently using TBX11K transform for all.
    chexpert_root=CHEXPERT_PATH,
    tbx_root=TBX11K_PATH
)

# Report readiness: dataset length and label order mapping.
print(f'\nCXRThreeClassDataset ready (relative-path mode): {len(train_dataset)} samples, label order={INDEX_TO_CLASS}')

Source pool counts (before split) [Pneumonia, TB, Normal]:
Pneumonia    4675
TB           4798
Normal       4178

CXRThreeClassDataset ready (relative-path mode): 13651 samples, label order={0: 'Pneumonia', 1: 'TB', 2: 'Normal'}
